In [ ]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict

#定义输入格式
class InputState(TypedDict):
    question: str

#定义输出格式
class OutputState(TypedDict):
    answer: str

# 将InputState和OutputState 这两个TypedDict 类型合并成一个字典类型
class OverallState(InputState, OutputState):
    pass


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

In [ ]:
from langgraph.graph import StateGraph
from typing_extensions import TypedDict, Optional
from langgraph.graph import START, END

# 定义输入的模式
class InputState(TypedDict):
    question: str
    llm_answer: Optional[str]  # 表示 answer 可以是 str 类型，也可以是 None

# 定义输出的模式
class OutputState(TypedDict):
    answer: str

# 将 InputState 和 OutputState 这两个 TypedDict 类型合并成一个更全面的字典类型。
class OverallState(InputState, OutputState):
    pass

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import getpass

def llm_node(state:InputState):
    messages = [
        {"role":"system","content":"你是一位乐于助人的智能小助理"},
        {"role":"user","content":state["question"]}
    ]

    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

    response=llm.invoke(messages)
    return {"llm_answer":response.content}

def action_node(state:InputState):
    messages = [
        {"role":"system","content":"无论你接收到什么语言的文本，请翻译成英语"},
        {"role":"user","content":state["llm_answer"]}
    ]

    llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

    response=llm.invoke(messages)
    return {"answer":response.content}

In [ ]:
builder = StateGraph(OverallState,input=InputState,output=OutputState)

#添加节点
builder.add_node("llm_node",llm_node)
builder.add_node("action_node",action_node)

#添加边
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node","action_node")
builder.add_edge("action_node",END)

#编译
graph=builder.compile()

In [ ]:
final_answer=graph.invoke({"question":"你好，请详细介绍一下你自己"})
print(final_answer)

In [ ]:
print(final_answer["answer"])

In [ ]:
final_answer= graph.invoke({"question":"请问什么是人工智能"})
print(final_answer["answer"])

In [ ]:
print(final_answer["answer"])